# ytdlp Colab Downloader

YouTube URL을 `wav`, `mp3`, `mp4`로 다운로드하고 Google Drive에 저장합니다.

위에서부터 셀을 순서대로 실행하세요.

## 1. Google Drive 연결

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. yt-dlp, ffmpeg 설치

In [ ]:
!python -m pip install -U yt-dlp
!apt-get update -qq
!apt-get install -y -qq ffmpeg nodejs
!yt-dlp --version
!node --version
!ffmpeg -version | head -1

## 3. 다운로드 설정 및 실행

In [ ]:
# @title
import os
import shlex
import subprocess
from pathlib import Path

#@markdown ---
#@markdown #### 다운로드 설정
#@markdown 다운로드 할 유튜브 URL을 입력하세요. 여러 개는 줄바꿈이나 쉼표로 구분합니다.
url_text = "https://www.youtube.com/watch?v=-upCeSPxhi4" #@param {type:"string"}

#@markdown 저장 포맷을 선택하세요.
output_format = "wav" #@param ["wav", "mp3", "mp4"]

#@markdown Google Drive 저장 폴더입니다.
output_folder = "/content/drive/MyDrive/ytdlp-downloads" #@param {type:"string"}
#@markdown ---

SUPPORTED_FORMATS = {"wav", "mp3", "mp4"}

def normalize_urls(urls):
    normalized = []
    seen = set()

    for value in urls:
        for url in str(value).split(","):
            url = url.strip()
            if url and url not in seen:
                normalized.append(url)
                seen.add(url)

    return normalized

def build_ytdlp_command(url, media_format, output_dir):
    output_template = str(Path(output_dir) / "%(title)s [%(id)s].%(ext)s")

    if media_format == "mp4":
        format_args = [
            "-f", "bv*+ba/b",
            "--merge-output-format", "mp4",
        ]
    else:
        format_args = [
            "-x",
            "--audio-format", media_format,
            "-f", "bestaudio/best",
        ]

    return [
        "yt-dlp",
        *format_args,
        "--force-overwrites",
        "--js-runtimes", "node",
        "-o", output_template,
        url,
    ]

def download_all(urls, media_format, output_dir):
    if media_format not in SUPPORTED_FORMATS:
        raise ValueError("FORMAT은 wav, mp3, mp4 중 하나여야 합니다.")

    unique_urls = normalize_urls(urls)
    if not unique_urls:
        raise ValueError("다운로드할 URL이 없습니다.")

    os.makedirs(output_dir, exist_ok=True)

    print(f"총 URL 수: {len(urls)}")
    print(f"중복 제거 후 URL 수: {len(unique_urls)}")
    print(f"다운로드 포맷: {media_format}")
    print(f"저장 위치: {output_dir}")

    failed = []
    for index, url in enumerate(unique_urls, start=1):
        print("\n------------------------------")
        print(f"[{index}/{len(unique_urls)}] 다운로드 시작")
        print(url)

        command = build_ytdlp_command(url, media_format, output_dir)
        print("실행 명령:", " ".join(shlex.quote(part) for part in command))
        result = subprocess.run(
            command,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        if result.stdout:
            print(result.stdout)

        if result.returncode == 0:
            print(f"[{index}/{len(unique_urls)}] 완료")
        else:
            print(f"[{index}/{len(unique_urls)}] 실패, 종료 코드: {result.returncode}")
            failed.append(url)

    print("\n전체 작업 완료")
    if failed:
        print("실패한 URL:")
        for url in failed:
            print(url)

URLS = normalize_urls(url_text.splitlines())
FORMAT = output_format
DRIVE_OUTPUT_DIR = output_folder.strip()

download_all(URLS, FORMAT, DRIVE_OUTPUT_DIR)

## 5. 저장된 파일 확인

In [ ]:
!find "$DRIVE_OUTPUT_DIR" -maxdepth 1 -type f | sort